# Pipeline End-to-End: YOLOv8 Cropping + Training VGG16 + Analisis Warna K-Means (Kategori Tas — 3 Kelas)
**Proyek Riset Sistem Pakar & Computer Vision — Prediksi Tren Fashion dari Media Sosial**  
Program Studi Informatika - Universitas Udayana

Notebook ini mengimplementasikan alur kerja (*end-to-end pipeline*) lengkap untuk kategori **TAS** (`tote_bag`, `handbag`, `sling_bag`):

### Alur Pipeline Utama:
1. **Tahap 1 — Setup & Konfigurasi Lingkungan:** Instalasi dan pemuatan pustaka (`ultralytics` YOLOv8, `torch`, `torchvision` VGG16, `scikit-learn` K-Means, `OpenCV`, `Pillow`, `Matplotlib`).
2. **Tahap 2 — Deteksi & Auto-Cropping Dataset dengan YOLOv8:**
   - Memuat dataset mentah (`dataset/tas/...`).
   - Mendeteksi area objek fesyen menggunakan **YOLOv8 (`yolov8n.pt`)** dan memotong (*crop*) bagian objek yang relevan.
   - Menyesuaikan ukuran potongan menjadi ukuran input standar VGG16 (**`224x224` piksel**) dan menyimpannya ke folder `dataset_cropped/tas/`.
3. **Tahap 3 — Data Loader & Augmentasi:** Membagi data hasil crop YOLOv8 menjadi **Train (70%)**, **Validation (15%)**, dan **Test (15%)**.
4. **Tahap 4 — Training Model Transfer Learning VGG16:**
   - Melatih arsitektur **VGG16** (*Pre-trained ImageNet Backbone* + *Custom Classifier Head*) pada data yang sudah di-crop oleh YOLOv8.
   - Menyimpan bobot model terbaik ke `best_vgg16_tas.pth`.
   - Evaluasi performa dengan kurva Loss/Accuracy, *Confusion Matrix*, dan *Classification Report* (*Precision, Recall, F1-Score*).
5. **Tahap 5 — Uji Coba Model (Inference Pipeline: YOLOv8 Crop $ightarrow$ Prediksi VGG16 $ightarrow$ Analisis Warna K-Means):**
   - Citra uji dideteksi dan di-*crop* terlebih dahulu menggunakan **YOLOv8** lalu di-*resize* ke ukuran input model (`224x224`).
   - Potongan citra diprediksi kelasnya menggunakan model **VGG16** yang telah dilatih.
   - Objek hasil crop dianalisis komposisi warna dominannya menggunakan algoritma **K-Means Clustering** (beserta kode HEX, nama warna fesyen, dan persentase proporsi warna).


## 1. Instalasi & Import Pustaka


In [ ]:
# Pastikan pustaka ultralytics (YOLOv8), opencv, dan scikit-learn telah terpasang
# Hapus tanda komentar pada baris di bawah jika dijalankan di Google Colab / lingkungan baru:
# !pip install -q ultralytics opencv-python-headless scikit-learn tqdm seaborn matplotlib

import os
import time
import copy
import random
import shutil
import zipfile
import urllib.request
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageDraw
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision
from torchvision import transforms, datasets
from torchvision.models import vgg16, VGG16_Weights

from ultralytics import YOLO
from sklearn.cluster import KMeans
from sklearn.metrics import classification_report, confusion_matrix

# Konfigurasi Seed untuk Reprodusibilitas
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"PyTorch Version     : {torch.__version__}")
print(f"Torchvision Version : {torchvision.__version__}")

In [ ]:
# Deteksi perangkat akselerasi perangkat keras (GPU CUDA / Apple MPS / CPU)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Hardware Acceleration : NVIDIA GPU ({torch.cuda.get_device_name(0)})")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Hardware Acceleration : Apple Silicon GPU (MPS)")
else:
    device = torch.device("cpu")
    print("Hardware Acceleration : CPU Mode")

CATEGORY_NAME = "tas"
INPUT_SIZE = (224, 224)  # Ukuran standar input model VGG16

## 2. Akuisisi Dataset & Auto-Cropping Menggunakan YOLOv8
Pada tahap ini:
1. Dataset diunduh dan diekstrak ke folder `dataset/tas`.
2. Model **YOLOv8 (`yolov8n.pt`)** dijalankan pada setiap foto untuk mendeteksi objek fesyen dan memotong (*crop*) bagian objeknya saja.
3. Hasil potongan disimpan ke `dataset_cropped/tas` dengan ukuran standar **`224x224` piksel** agar siap dilatih oleh VGG16.


In [ ]:
# Unduh & ekstrak dataset mentah secara otomatis jika belum tersedia
RAW_DATASET_DIR = "dataset"
RAW_CATEGORY_DIR = os.path.join(RAW_DATASET_DIR, CATEGORY_NAME)
CROPPED_DATASET_DIR = "dataset_cropped"
CROPPED_CATEGORY_DIR = os.path.join(CROPPED_DATASET_DIR, CATEGORY_NAME)

ZIP_URL = "http://202.10.47.34/slides/dataset_fashion_labeled.zip"
LOCAL_ZIP = "dataset_fashion_labeled.zip"

if not os.path.exists(RAW_CATEGORY_DIR):
    if not os.path.exists(LOCAL_ZIP):
        print(f"Mengunduh dataset dari {ZIP_URL}...")
        urllib.request.urlretrieve(ZIP_URL, LOCAL_ZIP)
        print("Unduhan selesai!")
    
    print("Mengekstrak berkas dataset...")
    with zipfile.ZipFile(LOCAL_ZIP, "r") as zf:
        zf.extractall(RAW_DATASET_DIR)
    print("Ekstraksi dataset selesai!")

print(f"Path Dataset Mentah: {RAW_CATEGORY_DIR}")
for cls_name in sorted(os.listdir(RAW_CATEGORY_DIR)):
    cls_path = os.path.join(RAW_CATEGORY_DIR, cls_name)
    if os.path.isdir(cls_path):
        n_imgs = len([f for f in os.listdir(cls_path) if f.lower().endswith((".jpg", ".jpeg", ".png"))])
        print(f"  - Kelas '{cls_name}': {n_imgs} gambar")

In [ ]:
# Muat model pre-trained YOLOv8 (otomatis mengunduh yolov8n.pt jika belum ada)
yolo_model = YOLO("yolov8n.pt")
print("Model YOLOv8 berhasil dimuat!")

# Kelas COCO pada YOLOv8 yang relevan:
# 0: person, 24: backpack, 26: handbag, 28: suitcase, 27: tie
BAG_COCO_CLASSES = [24, 26, 28]
PERSON_COCO_CLASS = 0

def detect_and_crop_yolov8(image_pil, category_mode="tas", target_size=(224, 224), conf_thres=0.20):
    """
    Mendeteksi objek fesyen menggunakan YOLOv8, melakukan cropping pada bagian yang sesuai,
    dan mengubah ukurannya menjadi target_size (224x224) untuk input VGG16.
    Mengembalikan:
      - cropped_resized_pil : PIL.Image (ukuran 224x224)
      - bbox_coords         : tuple (x1, y1, x2, y2) pada citra asli
      - detection_info      : dict informasi deteksi (label YOLO, confidence, metode crop)
    """
    img_rgb = np.array(image_pil.convert("RGB"))
    h, w, _ = img_rgb.shape

    results = yolo_model.predict(source=img_rgb, conf=conf_thres, verbose=False)
    boxes = results[0].boxes

    selected_box = None
    det_label = "center_fallback"
    det_conf = 1.0

    if boxes is not None and len(boxes) > 0:
        xyxy = boxes.xyxy.cpu().numpy()
        confs = boxes.conf.cpu().numpy()
        clss = boxes.cls.cpu().numpy().astype(int)

        if category_mode == "tas":
            # Prioritas 1: Deteksi langsung objek tas (handbag, backpack, suitcase)
            bag_indices = [i for i, c in enumerate(clss) if c in BAG_COCO_CLASSES]
            if len(bag_indices) > 0:
                # Pilih tas dengan kombinasi confidence dan ukuran area terbaik
                best_idx = max(bag_indices, key=lambda i: confs[i] * ((xyxy[i][2]-xyxy[i][0])*(xyxy[i][3]-xyxy[i][1])))
                x1, y1, x2, y2 = xyxy[best_idx]
                selected_box = (x1, y1, x2, y2)
                det_label = f"yolo_{yolo_model.names[clss[best_idx]]}"
                det_conf = float(confs[best_idx])
            else:
                # Prioritas 2: Jika tas dipakai oleh person, ambil area tengah-bawah person (tempat tas berada)
                person_indices = [i for i, c in enumerate(clss) if c == PERSON_COCO_CLASS]
                if len(person_indices) > 0:
                    best_idx = max(person_indices, key=lambda i: (xyxy[i][2]-xyxy[i][0])*(xyxy[i][3]-xyxy[i][1]))
                    px1, py1, px2, py2 = xyxy[best_idx]
                    pw, ph = (px2 - px1), (py2 - py1)
                    # Fokus pada bagian torso hingga pinggul/tangan pembawa tas
                    x1 = max(0, px1 - 0.05 * pw)
                    x2 = min(w, px2 + 0.05 * pw)
                    y1 = max(0, py1 + 0.18 * ph)
                    y2 = min(h, py1 + 0.90 * ph)
                    selected_box = (x1, y1, x2, y2)
                    det_label = "yolo_person_bag_region"
                    det_conf = float(confs[best_idx])

        else:  # category_mode == "baju"
            # Prioritas 1: Deteksi person dan crop area pakaian (torso/badan, membuang bagian atas kepala & background luar)
            person_indices = [i for i, c in enumerate(clss) if c == PERSON_COCO_CLASS]
            if len(person_indices) > 0:
                best_idx = max(person_indices, key=lambda i: (xyxy[i][2]-xyxy[i][0])*(xyxy[i][3]-xyxy[i][1]))
                px1, py1, px2, py2 = xyxy[best_idx]
                pw, ph = (px2 - px1), (py2 - py1)
                # Potong mulai dari bawah kepala/leher (12% dari atas box person) hingga 92% tinggi person
                x1 = max(0, px1 - 0.03 * pw)
                x2 = min(w, px2 + 0.03 * pw)
                y1 = max(0, py1 + 0.12 * ph)
                y2 = min(h, py1 + 0.92 * ph)
                selected_box = (x1, y1, x2, y2)
                det_label = "yolo_person_apparel"
                det_conf = float(confs[best_idx])
            else:
                # Prioritas 2: Ambil objek terbesar yang terdeteksi
                best_idx = int(np.argmax(confs))
                selected_box = tuple(xyxy[best_idx])
                det_label = f"yolo_{yolo_model.names[clss[best_idx]]}"
                det_conf = float(confs[best_idx])

    # Fallback: Jika YOLOv8 tidak menemukan bounding box (misal foto flat-lay produk murni), gunakan Center Crop 85%
    if selected_box is None:
        margin_x = int(0.075 * w)
        margin_y = int(0.075 * h)
        selected_box = (margin_x, margin_y, w - margin_x, h - margin_y)
        det_label = "center_object_crop"
        det_conf = 1.0

    # Tambahkan sedikit padding kontekstual (5%) dan pastikan koordinat valid
    x1, y1, x2, y2 = selected_box
    bw, bh = (x2 - x1), (y2 - y1)
    pad_x, pad_y = 0.04 * bw, 0.04 * bh
    x1 = int(max(0, x1 - pad_x))
    y1 = int(max(0, y1 - pad_y))
    x2 = int(min(w, x2 + pad_x))
    y2 = int(min(h, y2 + pad_y))

    # Pastikan ukuran box minimal 32x32 piksel
    if (x2 - x1) < 32 or (y2 - y1) < 32:
        x1, y1, x2, y2 = 0, 0, w, h

    cropped_pil = image_pil.crop((x1, y1, x2, y2)).convert("RGB")
    cropped_resized_pil = cropped_pil.resize(target_size, Image.Resampling.LANCZOS)

    detection_info = {
        "method": det_label,
        "confidence": det_conf,
        "bbox": (x1, y1, x2, y2),
        "orig_size": (w, h),
        "crop_size": target_size
    }
    return cropped_resized_pil, (x1, y1, x2, y2), detection_info

In [ ]:
# Jalankan proses Cropping YOLOv8 ke seluruh gambar di dataset dan simpan ke dataset_cropped/
os.makedirs(CROPPED_CATEGORY_DIR, exist_ok=True)

crop_stats = {"total": 0, "yolo_detected": 0, "fallback_crop": 0}
class_folders = sorted([d for d in os.listdir(RAW_CATEGORY_DIR) if os.path.isdir(os.path.join(RAW_CATEGORY_DIR, d))])

print(f"Memulai proses deteksi & cropping YOLOv8 untuk kategori '{CATEGORY_NAME}'...")
for cls_name in class_folders:
    src_cls_dir = os.path.join(RAW_CATEGORY_DIR, cls_name)
    dst_cls_dir = os.path.join(CROPPED_CATEGORY_DIR, cls_name)
    os.makedirs(dst_cls_dir, exist_ok=True)

    img_files = sorted([f for f in os.listdir(src_cls_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))])
    for fname in tqdm(img_files, desc=f"Cropping [{cls_name}]"):
        src_path = os.path.join(src_cls_dir, fname)
        dst_path = os.path.join(dst_cls_dir, fname)

        try:
            with Image.open(src_path) as img:
                cropped_img, bbox, info = detect_and_crop_yolov8(
                    img, category_mode=CATEGORY_NAME, target_size=INPUT_SIZE
                )
                cropped_img.save(dst_path, quality=95)
                crop_stats["total"] += 1
                if info["method"].startswith("yolo_"):
                    crop_stats["yolo_detected"] += 1
                else:
                    crop_stats["fallback_crop"] += 1
        except Exception as e:
            print(f"Gagal memproses {src_path}: {e}")

print("\nRingkasan Hasil Cropping YOLOv8:")
print(f"  - Total Gambar Diproses  : {crop_stats['total']}")
print(f"  - Terdeteksi BBox YOLOv8 : {crop_stats['yolo_detected']} ({crop_stats['yolo_detected']/max(1, crop_stats['total'])*100:.1f}%)")
print(f"  - Center Object Fallback : {crop_stats['fallback_crop']} ({crop_stats['fallback_crop']/max(1, crop_stats['total'])*100:.1f}%)")
print(f"  - Lokasi Dataset Crop    : {CROPPED_CATEGORY_DIR}")

In [ ]:
# Visualisasi Perbandingan Citra Asli (dengan Bounding Box YOLOv8) vs Hasil Crop (224x224)
sample_cls = class_folders[0]
sample_files = sorted(os.listdir(os.path.join(RAW_CATEGORY_DIR, sample_cls)))[:3]

fig, axes = plt.subplots(len(sample_files), 2, figsize=(10, 4 * len(sample_files)))
if len(sample_files) == 1:
    axes = np.expand_dims(axes, axis=0)

for idx, fname in enumerate(sample_files):
    raw_path = os.path.join(RAW_CATEGORY_DIR, sample_cls, fname)
    orig_img = Image.open(raw_path).convert("RGB")
    cropped_img, (x1, y1, x2, y2), info = detect_and_crop_yolov8(orig_img, category_mode=CATEGORY_NAME, target_size=INPUT_SIZE)

    # Gambar bounding box pada citra asli
    annotated_orig = orig_img.copy()
    draw = ImageDraw.Draw(annotated_orig)
    draw.rectangle([x1, y1, x2, y2], outline="#ef4444", width=5)

    axes[idx, 0].imshow(annotated_orig)
    axes[idx, 0].set_title(f"Citra Asli + BBox YOLOv8\n({info['method']}, conf={info['confidence']:.2f})", fontsize=11, fontweight="bold")
    axes[idx, 0].axis("off")

    axes[idx, 1].imshow(cropped_img)
    axes[idx, 1].set_title(f"Hasil Crop YOLOv8 ({INPUT_SIZE[0]}x{INPUT_SIZE[1]})\nKelas: {sample_cls}", fontsize=11, fontweight="bold")
    axes[idx, 1].axis("off")

plt.tight_layout()
plt.show()

## 3. Persiapan DataLoader dari Dataset Hasil Crop YOLOv8
Dataset yang digunakan untuk melatih **VGG16** diambil langsung dari direktori `dataset_cropped/` (berisi gambar yang sudah dipotong oleh YOLOv8 dan berukuran `224x224`).


In [ ]:
# Definisi Transformasi Data untuk Training & Validasi/Testing
train_transforms = transforms.Compose([
    transforms.Resize(INPUT_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=12),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transforms = transforms.Compose([
    transforms.Resize(INPUT_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Muat daftar sampel dari folder hasil crop YOLOv8
base_folder_dataset = datasets.ImageFolder(root=CROPPED_CATEGORY_DIR)
class_names = base_folder_dataset.classes
all_samples = base_folder_dataset.samples  # list of (path, label_idx)

# Pengacakan indeks secara terkontrol (Stratified/Seeded Split: 70% Train, 15% Val, 15% Test)
indices = list(range(len(all_samples)))
random.Random(SEED).shuffle(indices)

total_size = len(indices)
train_end = int(0.70 * total_size)
val_end = train_end + int(0.15 * total_size)

train_indices = indices[:train_end]
val_indices = indices[train_end:val_end]
test_indices = indices[val_end:]

class CroppedFashionDataset(Dataset):
    def __init__(self, samples, subset_indices, transform=None):
        self.samples = [samples[i] for i in subset_indices]
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

train_dataset = CroppedFashionDataset(all_samples, train_indices, transform=train_transforms)
val_dataset   = CroppedFashionDataset(all_samples, val_indices, transform=val_test_transforms)
test_dataset  = CroppedFashionDataset(all_samples, test_indices, transform=val_test_transforms)

BATCH_SIZE = 16
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Kelas Target ({len(class_names)}): {class_names}")
print(f"Pembagian Dataset Hasil Crop YOLOv8:")
print(f"  - Training   : {len(train_dataset)} gambar ({len(train_loader)} batch)")
print(f"  - Validation : {len(val_dataset)} gambar ({len(val_loader)} batch)")
print(f"  - Testing    : {len(test_dataset)} gambar ({len(test_loader)} batch)")

## 4. Arsitektur Transfer Learning VGG16 & Proses Training
Model **VGG16** menggunakan bobot *pre-trained* ImageNet. Layer konvolusi awal dibekukan (*frozen*), sedangkan blok konvolusi terakhir (`features[24:]`) dan *Classifier Head* dilatih (*fine-tuned*) agar mengenali tekstur dan bentuk potongan objek fesyen hasil deteksi YOLOv8.


In [ ]:
def build_vgg16_classifier(num_classes):
    model = vgg16(weights=VGG16_Weights.DEFAULT)

    # Bekukan layer ekstraksi fitur awal, buka blok konvolusi terakhir (features[24:]) untuk fine-tuning
    for param in model.features.parameters():
        param.requires_grad = False
    for param in model.features[24:].parameters():
        param.requires_grad = True

    # Custom Classifier Head
    model.classifier = nn.Sequential(
        nn.Linear(512 * 7 * 7, 1024),
        nn.BatchNorm1d(1024),
        nn.ReLU(inplace=True),
        nn.Dropout(p=0.5),
        nn.Linear(1024, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(inplace=True),
        nn.Dropout(p=0.4),
        nn.Linear(512, num_classes)
    )
    return model

num_classes = len(class_names)
model = build_vgg16_classifier(num_classes).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model VGG16 siap dilatih untuk {num_classes} kelas: {class_names}")
print(f"Total Parameter: {total_params:,} | Parameter Dilatih (Fine-Tuned): {trainable_params:,}")

In [ ]:
MODEL_SAVE_PATH = f"best_vgg16_{CATEGORY_NAME}.pth"

criterion = nn.CrossEntropyLoss()
trainable_params_list = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.AdamW(trainable_params_list, lr=1e-4, weight_decay=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

def train_vgg16_model(model, criterion, optimizer, scheduler, num_epochs=15):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_val_acc = 0.0

    history = {
        "train_loss": [], "train_acc": [],
        "val_loss": [], "val_acc": []
    }

    for epoch in range(num_epochs):
        print(f"Epoch {epoch + 1}/{num_epochs}")
        print("-" * 35)

        # --- Fase Training ---
        model.train()
        running_loss = 0.0
        running_corrects = 0

        for inputs, labels in tqdm(train_loader, desc="Training", leave=False):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        epoch_train_loss = running_loss / len(train_dataset)
        epoch_train_acc = running_corrects.double().item() / len(train_dataset)

        # --- Fase Validasi ---
        model.eval()
        val_loss = 0.0
        val_corrects = 0

        with torch.no_grad():
            for inputs, labels in tqdm(val_loader, desc="Validation", leave=False):
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * inputs.size(0)
                val_corrects += torch.sum(preds == labels.data)

        epoch_val_loss = val_loss / len(val_dataset)
        epoch_val_acc = val_corrects.double().item() / len(val_dataset)

        scheduler.step(epoch_val_loss)

        history["train_loss"].append(epoch_train_loss)
        history["train_acc"].append(epoch_train_acc)
        history["val_loss"].append(epoch_val_loss)
        history["val_acc"].append(epoch_val_acc)

        print(f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc*100:.2f}%")
        print(f"Val Loss  : {epoch_val_loss:.4f} | Val Acc  : {epoch_val_acc*100:.2f}%")

        # Simpan checkpoint model terbaik
        if epoch_val_acc >= best_val_acc:
            best_val_acc = epoch_val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            torch.save({
                "model_state_dict": best_model_wts,
                "class_names": class_names,
                "category": CATEGORY_NAME,
                "input_size": INPUT_SIZE
            }, MODEL_SAVE_PATH)
            print(f"--> Checkpoint terbaik disimpan ke '{MODEL_SAVE_PATH}' (Val Acc: {best_val_acc*100:.2f}%)")
        print()

    elapsed = time.time() - since
    print(f"Pelatihan selesai dalam {elapsed // 60:.0f}m {elapsed % 60:.0f}s | Best Val Acc: {best_val_acc*100:.2f}%")
    model.load_state_dict(best_model_wts)
    return model, history

# Jalankan Training Model VGG16
EPOCHS = 15
trained_model, history = train_vgg16_model(model, criterion, optimizer, scheduler, num_epochs=EPOCHS)

In [ ]:
# Visualisasi Kurva Loss & Akurasi serta Evaluasi pada Test Set
epochs_range = range(1, len(history["train_loss"]) + 1)
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, history["train_loss"], "-o", label="Training Loss", color="#0284c7")
plt.plot(epochs_range, history["val_loss"], "-s", label="Validation Loss", color="#ef4444")
plt.title(f"Loss vs Epochs ({CATEGORY_NAME.upper()})", fontsize=13, fontweight="bold")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, [a * 100 for a in history["train_acc"]], "-o", label="Training Accuracy", color="#0284c7")
plt.plot(epochs_range, [a * 100 for a in history["val_acc"]], "-s", label="Validation Accuracy", color="#10b981")
plt.title(f"Accuracy vs Epochs ({CATEGORY_NAME.upper()})", fontsize=13, fontweight="bold")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()

plt.tight_layout()
plt.savefig(f"training_curves_{CATEGORY_NAME}.png", dpi=300)
plt.show()

# Evaluasi pada Test Set
trained_model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = trained_model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print("=" * 65)
print(f"LAPORAN KLASIFIKASI MODEL VGG16 PADA TEST SET ({CATEGORY_NAME.upper()})")
print("=" * 65)
print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(7, 5.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.title(f"Confusion Matrix — VGG16 ({CATEGORY_NAME.upper()})", fontsize=13, fontweight="bold")
plt.xlabel("Prediksi Model", fontweight="bold")
plt.ylabel("Label Aktual (Ground Truth)", fontweight="bold")
plt.tight_layout()
plt.savefig(f"confusion_matrix_{CATEGORY_NAME}.png", dpi=300)
plt.show()

## 5. Uji Coba Model (End-to-End Inference: YOLOv8 Crop $ightarrow$ Prediksi VGG16 $ightarrow$ Analisis Warna K-Means)
Pada bagian ini, kita menguji model yang telah selesai dilatih dengan alur yang sama persis:
1. **Deteksi & Crop YOLOv8:** Foto input dideteksi oleh YOLOv8, lalu bagian objek yang terdeteksi di-*crop* dan di-*resize* ke ukuran input model VGG16 (`224x224`).
2. **Prediksi Model VGG16:** Gambar hasil crop diprediksi oleh model VGG16 (`best_vgg16_` + kategori) untuk menentukan jenis/sub-kategori item fesyen beserta *Confidence Score*-nya.
3. **Analisis Warna Dominan dengan K-Means Clustering:** Gambar hasil crop dianalisis menggunakan algoritma **K-Means (`sklearn.cluster.KMeans`)** untuk mengekstrak warna-warna dominan, persentase proporsinya, kode warna **HEX**, serta nama warna fesyennya.


In [ ]:
# Kamus Referensi Nama Warna Fesyen (RGB) untuk Pemetaan Hasil Centroid K-Means
FASHION_COLOR_PALETTE = {
    "Hitam (Black)": (25, 25, 25),
    "Putih (White)": (245, 245, 245),
    "Abu-Abu Tua (Charcoal)": (75, 75, 80),
    "Abu-Abu Muda (Light Grey)": (185, 185, 190),
    "Krem / Beige": (225, 208, 185),
    "Putih Gading (Ivory)": (248, 242, 228),
    "Cokelat Muda (Tan / Camel)": (193, 154, 107),
    "Cokelat Tua (Dark Brown)": (92, 64, 51),
    "Cokelat Kopi (Espresso)": (58, 40, 32),
    "Biru Dongker (Navy Blue)": (28, 46, 74),
    "Biru Denim (Denim Blue)": (70, 105, 145),
    "Biru Muda (Sky / Baby Blue)": (155, 195, 225),
    "Merah Marun (Burgundy)": (115, 28, 38),
    "Merah Menyala (Crimson Red)": (195, 35, 45),
    "Merah Muda (Dusty Pink)": (215, 155, 165),
    "Merah Muda Cerah (Blush Pink)": (242, 195, 205),
    "Hijau Sage (Sage Green)": (158, 175, 152),
    "Hijau Olive (Olive Green)": (105, 115, 65),
    "Hijau Zamrud (Emerald)": (25, 110, 75),
    "Kuning Mustard": (215, 165, 45),
    "Oranye Terracotta": (195, 95, 65),
    "Ungu Lavender / Lilac": (185, 165, 205)
}

def get_closest_fashion_color_name(rgb_tuple):
    """Mencari nama warna fesyen terdekat menggunakan jarak Euclidean pada ruang RGB."""
    r, g, b = rgb_tuple
    min_dist = float("inf")
    best_name = "Lainnya"
    for name, (cr, cg, cb) in FASHION_COLOR_PALETTE.items():
        dist = (r - cr) ** 2 + (g - cg) ** 2 + (b - cb) ** 2
        if dist < min_dist:
            min_dist = dist
            best_name = name
    return best_name

def analyze_dominant_colors_kmeans(cropped_pil, n_clusters=3):
    """
    Menganalisis warna dominan dari citra hasil crop YOLOv8 menggunakan K-Means Clustering.
    Mengambil area fokus tengah (center 75% dari hasil crop) agar meminimalkan piksel latar belakang di tepi potongan.
    """
    img_np = np.array(cropped_pil.convert("RGB"))
    h, w, _ = img_np.shape

    # Ambil fokus bagian dalam (75% tengah dari hasil crop objek) agar warna pakaian/tas lebih murni
    my, mx = int(0.125 * h), int(0.125 * w)
    core_region = img_np[my:h - my, mx:w - mx]
    pixels = core_region.reshape(-1, 3).astype(np.float32)

    kmeans = KMeans(n_clusters=n_clusters, random_state=SEED, n_init=10)
    labels = kmeans.fit_predict(pixels)
    centroids = kmeans.cluster_centers_.astype(int)

    # Hitung proporsi masing-masing cluster dan urutkan dari yang paling dominan
    counts = np.bincount(labels)
    total_pixels = len(labels)
    sorted_indices = np.argsort(-counts)

    color_results = []
    for idx in sorted_indices:
        rgb = tuple(int(v) for v in np.clip(centroids[idx], 0, 255))
        hex_code = "#{:02X}{:02X}{:02X}".format(*rgb)
        percentage = (counts[idx] / total_pixels) * 100.0
        color_name = get_closest_fashion_color_name(rgb)
        color_results.append({
            "rgb": rgb,
            "hex": hex_code,
            "percentage": percentage,
            "name": color_name
        })

    return color_results

In [ ]:
def load_trained_vgg16(checkpoint_path):
    """Memuat model VGG16 yang telah dilatih beserta daftar kelasnya."""
    checkpoint = torch.load(checkpoint_path, map_location=device)
    loaded_classes = checkpoint.get("class_names", class_names)
    loaded_model = build_vgg16_classifier(len(loaded_classes)).to(device)
    loaded_model.load_state_dict(checkpoint["model_state_dict"])
    loaded_model.eval()
    return loaded_model, loaded_classes

def test_fashion_pipeline(image_source, checkpoint_path=MODEL_SAVE_PATH, n_colors=3):
    """
    Flow Uji Coba End-to-End:
      1. Baca foto (dari path lokal atau URL)
      2. Deteksi & Crop bagian objek menggunakan YOLOv8 sesuai ukuran input VGG16 (224x224)
      3. Prediksi kategori menggunakan model VGG16 yang telah dilatih
      4. Analisis warna dominan pada hasil crop menggunakan K-Means Clustering
      5. Tampilkan visualisasi lengkap 4 panel
    """
    # 1. Muat gambar
    if str(image_source).startswith(("http://", "https://")):
        req = urllib.request.Request(image_source, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as resp:
            orig_pil = Image.open(resp).convert("RGB")
    else:
        orig_pil = Image.open(image_source).convert("RGB")

    # 2. Deteksi & Crop menggunakan YOLOv8 sesuai ukuran input VGG16 (224x224)
    cropped_224_pil, (x1, y1, x2, y2), det_info = detect_and_crop_yolov8(
        orig_pil, category_mode=CATEGORY_NAME, target_size=INPUT_SIZE
    )

    # 3. Prediksi menggunakan model VGG16
    if os.path.exists(checkpoint_path):
        infer_model, infer_classes = load_trained_vgg16(checkpoint_path)
    else:
        infer_model, infer_classes = trained_model, class_names
        infer_model.eval()

    input_tensor = val_test_transforms(cropped_224_pil).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = infer_model(input_tensor)
        probs = torch.softmax(logits, dim=1)[0].cpu().numpy()

    top_idx = int(np.argmax(probs))
    pred_class = infer_classes[top_idx]
    pred_conf = float(probs[top_idx]) * 100.0

    # 4. Analisis Warna Dominan dengan K-Means Clustering
    dominant_colors = analyze_dominant_colors_kmeans(cropped_224_pil, n_clusters=n_colors)

    # 5. Visualisasi Hasil End-to-End (4 Panel)
    fig, axes = plt.subplots(1, 4, figsize=(18, 4.8))

    # Panel 1: Foto Asli + Bounding Box YOLOv8
    annotated_img = orig_pil.copy()
    draw = ImageDraw.Draw(annotated_img)
    draw.rectangle([x1, y1, x2, y2], outline="#ef4444", width=5)
    axes[0].imshow(annotated_img)
    axes[0].set_title(
        f"1. Deteksi YOLOv8\n({det_info['method']}, conf={det_info['confidence']:.2f})",
        fontsize=11, fontweight="bold"
    )
    axes[0].axis("off")

    # Panel 2: Hasil Crop YOLOv8 (224x224) Input VGG16
    axes[1].imshow(cropped_224_pil)
    axes[1].set_title(
        f"2. Hasil Crop ({INPUT_SIZE[0]}x{INPUT_SIZE[1]})\nPrediksi VGG16: {pred_class.upper()} ({pred_conf:.1f}%)",
        fontsize=11, fontweight="bold", color="#0f172a"
    )
    axes[1].axis("off")

    # Panel 3: Grafik Probabilitas Prediksi VGG16
    y_pos = np.arange(len(infer_classes))
    bars = axes[2].barh(y_pos, probs * 100.0, color="#0284c7", edgecolor="#0369a1")
    axes[2].set_yticks(y_pos)
    axes[2].set_yticklabels(infer_classes, fontweight="bold")
    axes[2].set_xlim(0, 105)
    axes[2].set_xlabel("Confidence (%)")
    axes[2].set_title("3. Probabilitas Kelas VGG16", fontsize=11, fontweight="bold")
    axes[2].grid(axis="x", linestyle="--", alpha=0.5)
    for bar, p in zip(bars, probs * 100.0):
        axes[2].text(min(p + 2, 95), bar.get_y() + bar.get_height()/2, f"{p:.1f}%", va="center", fontsize=9.5, fontweight="bold")

    # Panel 4: Palet Warna Dominan K-Means
    axes[3].set_title(f"4. Analisis Warna K-Means (Top {n_colors})", fontsize=11, fontweight="bold")
    y_cursor = 0.82
    for idx_c, c_info in enumerate(dominant_colors):
        rgb_norm = tuple(v / 255.0 for v in c_info["rgb"])
        # Kotak warna
        rect = plt.Rectangle((0.05, y_cursor - 0.12), 0.22, 0.18, facecolor=rgb_norm, edgecolor="#334155", linewidth=1.5, transform=axes[3].transAxes)
        axes[3].add_patch(rect)
        # Informasi teks warna
        axes[3].text(
            0.32, y_cursor - 0.03,
            f"#{idx_c+1}: {c_info['name']}\nHEX: {c_info['hex']} | Proporsi: {c_info['percentage']:.1f}%",
            transform=axes[3].transAxes,
            fontsize=10, fontweight="bold", va="center", color="#1e293b"
        )
        y_cursor -= 0.28
    axes[3].axis("off")

    plt.tight_layout()
    plt.show()

    # Cetak ringkasan teks
    print("=" * 65)
    print("HASIL INFERENSI PIPELINE (YOLOv8 -> VGG16 -> K-MEANS)")
    print("=" * 65)
    print(f"Sumber Citra        : {image_source}")
    print(f"Metode Crop YOLOv8  : {det_info['method']} (BBox: {det_info['bbox']}, Ukuran Input: {INPUT_SIZE})")
    print(f"Prediksi VGG16      : {pred_class} (Confidence: {pred_conf:.2f}%)")
    print("Warna Dominan (K-Means):")
    for i, c in enumerate(dominant_colors, 1):
        print(f"  {i}. {c['name']} ({c['hex']}) - RGB{c['rgb']} -> {c['percentage']:.2f}%")
    print("=" * 65)

    return {
        "predicted_class": pred_class,
        "confidence": pred_conf,
        "bbox": det_info["bbox"],
        "dominant_colors": dominant_colors
    }

In [ ]:
# --- DEMO 1: Uji Coba Model pada Sampel Gambar dari setiap Kelas ---
print("Menguji pipeline lengkap (YOLOv8 Crop -> VGG16 Predict -> K-Means Color) pada sampel foto asli:\n")

for cls_name in class_folders:
    raw_cls_dir = os.path.join(RAW_CATEGORY_DIR, cls_name)
    raw_files = sorted([f for f in os.listdir(raw_cls_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))])
    if len(raw_files) > 0:
        sample_raw_path = os.path.join(raw_cls_dir, raw_files[0])
        print(f">>> Ground Truth Kelas Asli: {cls_name.upper()}")
        _ = test_fashion_pipeline(sample_raw_path, checkpoint_path=MODEL_SAVE_PATH, n_colors=3)

In [ ]:
# --- DEMO 2: Cell Khusus untuk Mencoba Foto Baru (Path Lokal / URL / Upload Google Colab) ---
# Anda dapat mengganti CUSTOM_IMAGE_PATH dengan path file gambar Anda sendiri atau URL gambar dari internet:
CUSTOM_IMAGE_PATH = ""  # Contoh: "foto_uji_baru.jpg" atau "https://..."

if CUSTOM_IMAGE_PATH and (os.path.exists(CUSTOM_IMAGE_PATH) or CUSTOM_IMAGE_PATH.startswith(("http://", "https://"))):
    hasil_uji = test_fashion_pipeline(CUSTOM_IMAGE_PATH, checkpoint_path=MODEL_SAVE_PATH, n_colors=3)
else:
    print("Tip: Isi variabel CUSTOM_IMAGE_PATH di atas dengan path foto atau URL gambar yang ingin Anda uji.")
    print("Jika Anda menjalankan di Google Colab, Anda juga dapat mengunggah foto secara langsung dengan menghapus komentar kode di bawah ini:")
    print("# from google.colab import files")
    print("# uploaded = files.upload()")
    print("# for fn in uploaded.keys():")
    print("#     test_fashion_pipeline(fn, checkpoint_path=MODEL_SAVE_PATH, n_colors=3)")